# StructVerify Colab Runner

브랜치 클론, 환경설정, 레이어별 실행, 결과 저장, 별도 레포 push까지 한 번에 확인하는 노트북입니다.

In [ ]:
# ====== Colab 세션 환경변수 ======
REPO_URL = "https://github.com/<org>/<repo>.git"
BRANCH_NAME = "main"
RUN_VERSION = "v001"
SAVE_RESULTS = "1"
RESULTS_REPO_URL = ""  # 선택
RESULTS_REPO_BRANCH = "main"

# API 키가 필요하면 아래에 넣거나 Colab secret/환경변수로 주입하세요.
OPENAI_API_KEY = ""
KOSIS_API_KEY = ""


In [ ]:
%%bash
set -e
rm -rf 2026-NLP-04 || true
git clone --depth 1 --branch "$BRANCH_NAME" "$REPO_URL" 2026-NLP-04
cd 2026-NLP-04/backend
pip install -r requirements-dev.txt
pip install -e .


In [ ]:
import os
from pathlib import Path

os.environ["SV_REPO_URL"] = REPO_URL
os.environ["SV_BRANCH"] = BRANCH_NAME
os.environ["SV_VERSION"] = RUN_VERSION
os.environ["SV_SAVE_RESULTS"] = SAVE_RESULTS
os.environ["SV_RESULTS_REPO_URL"] = RESULTS_REPO_URL
os.environ["SV_RESULTS_REPO_BRANCH"] = RESULTS_REPO_BRANCH
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["KOSIS_API_KEY"] = KOSIS_API_KEY

%cd /content/2026-NLP-04/backend
!source scripts/env_session.sh && sv_print_env


## 전체 파이프라인 실행

In [ ]:
!bash scripts/run_pipeline.sh \
  --text "국내 과수 농가의 65세 이상 고령화 비율은 64.2%에 이른다" \
  --output-name pipeline_smoke


## 레이어별 실행

In [ ]:
!bash scripts/run_layer.sh extract --text "국내 과수 농가의 65세 이상 고령화 비율은 64.2%에 이른다" --output-name layer_extract
!bash scripts/run_layer.sh sir --text "국내 과수 농가의 65세 이상 고령화 비율은 64.2%에 이른다" --output-name layer_sir
!bash scripts/run_layer.sh claims --text "국내 과수 농가의 65세 이상 고령화 비율은 64.2%에 이른다" --output-name layer_claims
!bash scripts/run_layer.sh schema --text "국내 과수 농가의 65세 이상 고령화 비율은 64.2%에 이른다" --output-name layer_schema


## verify / explain 단독 테스트

In [ ]:
!bash scripts/run_layer.sh verify --claim-text "실업률 10%" --claimed-value 10 --official-value 7.2 --output-name layer_verify
!bash scripts/run_layer.sh explain --claim-text "실업률 10%" --claimed-value 10 --official-value 7.2 --output-name layer_explain


## 저장된 결과 확인

In [ ]:
from pathlib import Path
results_dir = Path("run_outputs") / f"{BRANCH_NAME}_{RUN_VERSION}"
print(results_dir)
for p in sorted(results_dir.glob("*.json")):
    print(p.name)


## 선택: 결과를 별도 레포로 push

In [ ]:
# RESULTS_REPO_URL 을 채운 경우만 실행
if RESULTS_REPO_URL:
    !bash scripts/push_results.sh
else:
    print("RESULTS_REPO_URL 이 비어 있어서 skip")
